# Teste de Microfone Offline:

In [ ]:
import pyaudio
import wave
import numpy as np
print(np.__version__)
from scipy.signal import resample_poly

In [ ]:
FORMAT = pyaudio.paInt16
CHANNELS = 1
RATE = 48000
TARGET_RATE = 16000

CHUNK = 1024
RECORD_SECONDS = 10

DEVICE_INDEX = 0   # replace this with your detected USB mic's index
WAVE_OUTPUT_FILENAME = "output.wav"

In [ ]:
audio = pyaudio.PyAudio()

stream = audio.open(
    format=FORMAT,
    channels=CHANNELS,
    rate=RATE,
    input=True,
    input_device_index=DEVICE_INDEX,
    frames_per_buffer=CHUNK
)

In [ ]:
print("Recording...")

frames = []

for i in range(0, int(RATE / CHUNK * RECORD_SECONDS)):
    data = stream.read(CHUNK, exception_on_overflow=False)
    frames.append(data)

print("Finished recording.")

stream.stop_stream()
stream.close()
audio.terminate()

audio_data = np.frombuffer(b''.join(frames), dtype=np.int16)

# Calcula os fatores para a reamostragem (de 48k para 16k é 1/3)
resampled_data = resample_poly(audio_data, TARGET_RATE, RATE)

# Converte de volta para int16
resampled_data = resampled_data.astype(np.int16)

wf = wave.open(WAVE_OUTPUT_FILENAME, 'wb')

# Define os parâmetros do ÁUDIO REAMOSTRADO
wf.setnchannels(CHANNELS)
wf.setsampwidth(audio.get_sample_size(FORMAT))
wf.setframerate(TARGET_RATE) # <-- MUDANÇA 1: Usar a taxa de amostragem de destino

# Escreve os dados REAMOSTRADOS no arquivo
# .tobytes() converte o array NumPy de volta para bytes
wf.writeframes(resampled_data.tobytes()) # <-- MUDANÇA 2: Usar os dados reamostrados

wf.close()

In [ ]:
!ls

In [ ]:
import IPython.display as ipd
ipd.Audio("output.wav")

# Teste Modelo:

In [ ]:
!ls

In [ ]:
import time
import numpy as np
import pyaudio
import librosa
import tflite_runtime.interpreter as tflite
from collections import deque
from scipy.signal import resample_poly
from gpiozero import LED
import wave

In [ ]:
LED_GO = LED(26)
LED_STOP = LED(6)

LED_ON = LED(27)
LED_OFF = LED(4)

LED_NO = LED(5)

In [ ]:
LED_GO.off()
LED_STOP.off()
LED_ON.off()
LED_OFF.off()
LED_NO.off()

In [ ]:
# Teste de LEDs:

LED_GO.on()
LED_STOP.on()
LED_ON.on()
LED_OFF.on()
LED_NO.on()

time.sleep(3)  # Mantém os LEDs acesos por 3 segundos

LED_GO.off()
LED_STOP.off()
LED_ON.off()
LED_OFF.off()
LED_NO.off()

Copie o modelo para o diretório atual

In [ ]:
!ls

In [ ]:
CLASS_NAMES = ['_silence_', '_unknown_', 'go', 'no', 'off', 'on', 'stop']

MODEL_PATH = "./model_mfcc_fp16.tflite"

In [ ]:
FORMAT = pyaudio.paInt16
CHANNELS = 1
RATE = 48000
TARGET_RATE = 16000

RECORD_SECONDS = 1
CHUNK = RATE * RECORD_SECONDS

DEVICE_INDEX = 0

In [ ]:
print("Carregando modelo TFLite...")

interpreter = tflite.Interpreter(model_path=MODEL_PATH)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("Modelo carregado com sucesso.")
print("Input shape:", input_details[0]['shape'])

In [ ]:
N_FFT = 512
FRAME_STEP = 256
N_MELS = 128
N_MFCCS = 40
F_MIN = 20
F_MAX = TARGET_RATE / 2

In [ ]:
def preprocess_audio(audio_data):
    """
    Recebe 1 segundo de áudio, extrai os MFCCs e formata
    para a entrada do modelo.
    """
    mfccs = librosa.feature.mfcc(
        y=audio_data,
        sr=TARGET_RATE,
        n_mfcc=N_MFCCS,
        n_fft=N_FFT,
        hop_length=FRAME_STEP,
        win_length=N_FFT,
        n_mels=N_MELS,
        fmin=F_MIN,
        fmax=F_MAX,
        center=False
    )
    
    mfccs = mfccs.T
    mfccs = np.expand_dims(mfccs, axis=-1)
    mfccs = np.expand_dims(mfccs, axis=0)
    
    return mfccs.astype(np.float32)

In [ ]:
audio = pyaudio.PyAudio()

WAVE_OUTPUT_FILENAME = "output.wav"

stream = audio.open(
    format=FORMAT,
    channels=CHANNELS,
    rate=RATE,
    input=True,
    input_device_index=DEVICE_INDEX,
    frames_per_buffer=CHUNK
)

print("Recording...")

frames = []

for i in range(0, int(RATE / CHUNK * RECORD_SECONDS)):
    data = stream.read(CHUNK, exception_on_overflow=False)
    frames.append(data)

print("Finished recording.")

stream.stop_stream()
stream.close()
audio.terminate()

audio_data = np.frombuffer(b''.join(frames), dtype=np.int16)

resampled_data = resample_poly(audio_data, TARGET_RATE, RATE)

resampled_data = resampled_data.astype(np.int16)

wf = wave.open(WAVE_OUTPUT_FILENAME, 'wb')
wf.setnchannels(CHANNELS)
wf.setsampwidth(audio.get_sample_size(FORMAT))
wf.setframerate(TARGET_RATE)
wf.writeframes(resampled_data.tobytes())

wf.close()

In [ ]:
import IPython.display as ipd
ipd.Audio("output.wav")

In [ ]:
resampled_data = resampled_data.astype(np.float32) / 32768.0

In [ ]:
max(resampled_data)

In [ ]:
min(resampled_data)

In [ ]:
preprocessed_data = preprocess_audio(resampled_data)

In [ ]:
preprocessed_data.shape

In [ ]:
import matplotlib.pyplot as plt

plt.imshow(preprocessed_data[0, :, :, 0].T, aspect='auto', origin='lower')

In [ ]:
print("Carregando modelo TFLite...")

interpreter = tflite.Interpreter(model_path=MODEL_PATH)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("Modelo carregado com sucesso.")
print("Input shape:", input_details[0]['shape'])

# Executa a inferência
interpreter.set_tensor(input_details[0]['index'], preprocessed_data)
interpreter.invoke()

output_data = interpreter.get_tensor(output_details[0]['index'])
prediction_index = np.argmax(output_data[0])
predicted_class = CLASS_NAMES[prediction_index]
confidence = np.max(output_data[0])

print(f"DETECÇÃO: {predicted_class} (Confiança: {confidence:.2f})")

# Teste Real-Time:

In [1]:
import time
import numpy as np
import pyaudio
import librosa
import tflite_runtime.interpreter as tflite
from collections import deque
from scipy.signal import resample_poly
from gpiozero import LED
from scipy.fftpack import dct

LED_GO = LED(26)
LED_STOP = LED(6)
LED_ON = LED(27)
LED_OFF = LED(4)
LED_NO = LED(5)

def preprocess_audio(audio_data):
    """
    Recebe 1 segundo de áudio, extrai os MFCCs e formata
    para a entrada do modelo, REPLICANDO o pipeline do TensorFlow.
    """
    
    # 1. Calcular o Mel Spectrogram (baseado em MAGNITUDE, não potência)
    # power=1.0 é crucial para corresponder a tf.abs(stft)
    mel_spectrogram = librosa.feature.melspectrogram(
        y=audio_data,
        sr=TARGET_RATE,
        n_fft=N_FFT,
        hop_length=FRAME_STEP,
        win_length=N_FFT,
        n_mels=N_MELS,
        fmin=F_MIN,
        fmax=F_MAX,
        center=False, # Corresponde ao tf.signal.stft (sem pre-padding)
        power=1.0 # <-- IMPORTANTE! O TF usou tf.abs(stft)
    )
    
    # 2. Aplicar a mesma escala de log do TensorFlow
    log_mel_spectrogram = np.log(mel_spectrogram + 1e-6)

    # 3. Calcular MFCCs a partir do log-mel (DCT-II)
    # Aplicamos a DCT no eixo dos Mels (axis=0)
    # O tf.signal.mfccs_from_log_mel_spectrograms aplica DCT-II
    mfccs = dct(log_mel_spectrogram, type=2, axis=0, norm='ortho')
    
    # 4. Pegar os N_MFCCS e transpor
    # O resultado de dct é (n_mels, time). Queremos (time, n_mfccs)
    mfccs = mfccs[:N_MFCCS, :] # Pegar os N_MFCCS primeiros
    mfccs = mfccs.T # Transpor para (time, n_mfccs)

    # 5. Formatar para o TFLite
    mfccs = np.expand_dims(mfccs, axis=-1)
    mfccs = np.expand_dims(mfccs, axis=0)
    
    return mfccs.astype(np.float32)

CLASS_NAMES = ['_silence_', '_unknown_', 'go', 'no', 'off', 'on', 'stop']
# MODEL_PATH = "./model_mfcc_fp16.tflite"
MODEL_PATH = "./model_mfcc_finetuned_fp16.tflite"

FORMAT = pyaudio.paInt16
CHANNELS = 1
RATE = 48000
TARGET_RATE = 16000
DEVICE_INDEX = 0

WINDOW_DURATION_S = 1.0
STEP_DURATION_S = 0.5

CHUNK = int(RATE * STEP_DURATION_S)

BUFFER_MAX_LEN = int(WINDOW_DURATION_S / STEP_DURATION_S)

N_FFT = 512
FRAME_STEP = 256
N_MELS = 128
N_MFCCS = 40
F_MIN = 20
F_MAX = TARGET_RATE / 2

audio = pyaudio.PyAudio()

stream = audio.open(
    format=FORMAT,
    channels=CHANNELS,
    rate=RATE,
    input=True,
    input_device_index=DEVICE_INDEX,
    frames_per_buffer=CHUNK
)

print("Carregando modelo TFLite...")

interpreter = tflite.Interpreter(model_path=MODEL_PATH)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("Modelo carregado com sucesso.")
print("Input shape:", input_details[0]['shape'])

print("\nIniciando inferência com janela deslizante... (Pressione Ctrl+C para parar)")

audio_buffer = deque(maxlen=BUFFER_MAX_LEN)

LAST_DETECTION_TIME = 0
COOLDOWN_S = 1.5

try:
    while True:
        # 1. Ler 0.5s de áudio (CHUNK)
        data = stream.read(CHUNK, exception_on_overflow=False)
        
        # 2. Adicionar ao buffer
        audio_buffer.append(data)

        # 3. Se o buffer ainda não estiver cheio (primeiro 1s),
        # pular a inferência e continuar enchendo.
        if len(audio_buffer) < BUFFER_MAX_LEN:
            continue
            
        # 4. Concatenar os 2 blocos de 0.5s para formar 1.0s de áudio
        full_buffer_bytes = b''.join(audio_buffer)
        
        # 5. Converter buffer para numpy (int16) - 1.0s a 48kHz
        audio_data = np.frombuffer(full_buffer_bytes, dtype=np.int16)

        # 6. Resample de 48kHz para 16kHz
        resampled_data = resample_poly(audio_data, TARGET_RATE, RATE)

        # 7. Normalizar para float32
        resampled_data_float = resampled_data.astype(np.float32) / 32768.0

        # 8. Pré-processar (MFCCs)
        preprocessed_data = preprocess_audio(resampled_data_float)
        
        # 9. Executar a inferência
        interpreter.set_tensor(input_details[0]['index'], preprocessed_data)
        interpreter.invoke()

        # 10. Obter resultados
        output_data = interpreter.get_tensor(output_details[0]['index'])
        prediction_index = np.argmax(output_data[0])
        predicted_class = CLASS_NAMES[prediction_index]
        confidence = np.max(output_data[0])

        # 11. Aplicar "Cooldown" para evitar detecções duplicadas
        current_time = time.time()
        
        if predicted_class not in ['_silence_', '_unknown_'] and (current_time - LAST_DETECTION_TIME > COOLDOWN_S):
            print(f"DETECÇÃO: {predicted_class} (Confiança: {confidence:.2f})")

            # Acionar LEDs conforme a detecção
            if predicted_class == 'go':
                LED_GO.on()
                time.sleep(1)
                LED_GO.off()
            elif predicted_class == 'stop':
                LED_STOP.on()
                time.sleep(1)
                LED_STOP.off()
            elif predicted_class == 'on':
                LED_ON.on()
                time.sleep(1)
                LED_ON.off()
            elif predicted_class == 'off':
                LED_OFF.on()
                time.sleep(1)
                LED_OFF.off()
            elif predicted_class == 'no':
                LED_NO.on()
                time.sleep(1)
                LED_NO.off()
                
            LAST_DETECTION_TIME = current_time # Reinicia o cooldown

except KeyboardInterrupt:
    print("\nParando a gravação.")

finally:
    print("Liberando recursos...")
    stream.stop_stream()
    stream.close()
    audio.terminate()
    print("Pronto.")


Parando a gravação.
Liberando recursos...
Pronto.
